# 지난 기록으로 돌아가기 · time travel

강의 오후 ④ 가운데 (Academy 모듈 3 · L5 Time travel)

| 칸 | 보여 주는 것 |
| --- | --- |
| ① 지난 기록 | `getStateHistory` 로 thread 의 checkpoint 를 모두 본다 |
| ② 다시 돌리기 | 지난 checkpoint 의 config 로 `stream(null)` 을 부르면 그 자리부터 다시 돈다 |
| ③ 갈라지기 | 지난 checkpoint 에 `updateState` 로 값을 넣으면 새 갈래가 생긴다. 그 갈래로 이어 간다 |

checkpointer 는 단계마다 State 를 저장합니다. 오후 ② · ③ 에서는 그중 **마지막** 것만 썼습니다(`getState`). 이번에는 지난 것들을 씁니다.

In [ ]:
import "npm:@langchain/core@1.2.14";
import { Annotation, StateGraph, START, END, MemorySaver } from "npm:@langchain/langgraph@1.4.18";

// 접속 정보: 실습과 같은 lab/.env 를 읽는다 (OpenRouter · qwen/qwen3-32b). 키는 화면에 찍지 않는다
const ENV: Record<string, string> = {};
try {
  for (const line of Deno.readTextFileSync("../lab/.env").split("\n")) {
    const m = line.match(/^\s*([A-Z_]+)\s*=\s*(.*)\s*$/);
    if (m) ENV[m[1]] = m[2];
  }
} catch { /* lab/.env 가 없으면 아래 기본값 */ }
const BASE_URL = ENV.LLM_BASE_URL ?? "https://openrouter.ai/api/v1";
const API_KEY  = ENV.LLM_API_KEY  ?? "";
const MODEL    = ENV.LLM_MODEL    ?? "qwen/qwen3-32b";
const OPTIONS  = { temperature: 0 };   // 시연이 매번 같게
console.log({ BASE_URL, MODEL, 키: API_KEY ? "있음" : "없음 — lab/.env 에 LLM_API_KEY 를 넣는다" });

// LangSmith (강사 시연용): demo/.env 에 키가 있으면 실행 기록(trace)이 LangSmith 에 남는다. 실습은 lab/.env 를 써서 남지 않는다
import { traceable } from "npm:langsmith@0.11.1/traceable";
try {
  for (const line of Deno.readTextFileSync(".env").split("\n")) {
    const m = line.match(/^\s*(LANGSMITH_[A-Z_]+)\s*=\s*(?:"([^"]*)"|([^#\s]*))/);   // "따옴표 값" 은 그대로, 아니면 # 주석 앞까지
    const value = m?.[2] ?? m?.[3];
    if (m && value) Deno.env.set(m[1], value);
  }
} catch { /* demo/.env 가 없으면 기록하지 않는다 */ }
if (!Deno.env.get("LANGSMITH_API_KEY")) Deno.env.set("LANGSMITH_TRACING", "false");   // 키가 없으면 끈다
// 모델을 부르는 함수를 감싸면 trace 에 "모델 호출" 한 칸으로 남는다. 기록을 안 켜면 그냥 원래 함수로 돈다
const asLLM = <F extends (...a: any[]) => Promise<any>>(fn: F, name: string) => traceable(fn, {
  name, run_type: "llm",
  processInputs: (i: any) => ({ messages: i.args ? i.args[0] : i.input }),
  processOutputs: (o: any) => ({ choices: [{ message: typeof o.outputs === "string" ? { role: "assistant", content: o.outputs } : o }] }),
});
console.log({ LangSmith: Deno.env.get("LANGSMITH_TRACING") === "true" && Deno.env.get("LANGSMITH_API_KEY") ? `기록함 · project ${Deno.env.get("LANGSMITH_PROJECT") ?? "default"}` : "기록 안 함 (demo/.env)" });

// 그래프 그림: 노드·엣지 이름을 mermaid.ink 로 보내 PNG 로 받아 셀 출력에 띄운다. 받지 못하면 mermaid 글자로 찍는다
async function showGraph(graph: any, options: { xray?: boolean } = {}) {
  const drawable = await graph.getGraphAsync(options);   // xray: true 면 subgraph 안의 노드까지 그린다
  try {
    const png = await drawable.drawMermaidPng();
    await Deno.jupyter.display(Deno.jupyter.image(new Uint8Array(await png.arrayBuffer())));
  } catch {
    console.log(drawable.drawMermaid());
  }
}

## 준비: 튜터 Agent

오후 ③ 과 같은 튜터 Agent 에 checkpointer 를 답니다.

In [ ]:
// 오전에 만든 chat() 그대로 — 다시 보낼 칸(role · content · tool_calls)만 돌려준다
async function callModel(messages: any[], tools?: any[]) {
  const res = await fetch(`${BASE_URL}/chat/completions`, {
    method: "POST",
    headers: { "Content-Type": "application/json", Authorization: `Bearer ${API_KEY}` },
    body: JSON.stringify({ model: MODEL, messages, tools, ...OPTIONS }),
  });
  const { role, content, tool_calls } = (await res.json()).choices[0].message;
  return { role, content: content?.trim() ?? null, tool_calls };
}
const chat = asLLM(callModel, "chat");   // LangSmith 에 모델 호출로 남긴다

// 도구 하나: 단어 뜻과 쉬운 말 찾기 (작은 사전이라 시연이 매번 같다)
const DICT: Record<string, { 뜻: string; 쉬운말: string }> = {
  ubiquitous: { 뜻: "어디에나 있는", 쉬운말: "everywhere" },
  mitigate:   { 뜻: "완화하다, 줄이다", 쉬운말: "reduce" },
  meticulous: { 뜻: "꼼꼼한", 쉬운말: "careful" },
  detrimental:{ 뜻: "해로운", 쉬운말: "harmful" },
};
const tools = [{
  type: "function",
  function: {
    name: "lookup_word",
    description: "영어 단어 하나의 뜻과, 바꿔 쓸 쉬운 말을 사전에서 찾는다",
    parameters: { type: "object", properties: { word: { type: "string", description: "찾을 영어 단어 하나" } }, required: ["word"] },
  },
}];
function lookupWord(word: string) {
  const w = word.toLowerCase().trim();
  return DICT[w] ? { word: w, ...DICT[w] } : { word: w, error: "사전에 없는 단어" };
}

const SYSTEM = { role: "system", content: "너는 IELTS 쓰기를 돕는 영어 튜터다. 단어 뜻은 추측하지 말고 lookup_word 로 확인한다. 한국어로 짧게 답한다." };
const ask = (q: string) => ({ messages: [SYSTEM, { role: "user", content: q }] });
// 메시지를 보기 좋게 찍는다 (tutor-stream 노트북의 pretty 그대로)
const TITLE: Record<string, string> = { system: "System Message", user: "Human Message", assistant: "Ai Message", tool: "Tool Message" };
function pretty(m: any) {
  const title = ` ${TITLE[m.role]} `;
  const left = Math.floor((60 - title.length) / 2);
  console.log("=".repeat(left) + title + "=".repeat(60 - title.length - left));
  if (m.tool_calls?.length) {
    const calls = m.tool_calls.map((c: any) => `${c.function.name}(${c.function.arguments})`);
    console.log("Tool Calls:", calls.join(", "));
  } else console.log(m.content);
}

In [ ]:
// 오후 ① 의 튜터 Agent: tutor ⇢ tools → tutor
const State = Annotation.Root({
  messages: Annotation<any[]>({ reducer: (prev, next) => [...prev, ...next], default: () => [] }),
});
async function tutor(state: typeof State.State) {
  return { messages: [await chat(state.messages, tools)] };
}
function route(state: typeof State.State) {
  return state.messages.at(-1).tool_calls?.length ? "tools" : END;
}
function toolsNode(state: typeof State.State) {
  const calls = state.messages.at(-1).tool_calls;
  console.log(`▶ tools   ${calls.map((c: any) => c.function.arguments).join(" · ")}`);
  return { messages: calls.map((c: any) => ({
    role: "tool", tool_call_id: c.id, content: JSON.stringify(lookupWord(JSON.parse(c.function.arguments).word)),
  })) };
}
const agent = new StateGraph(State)
  .addNode("tutor", tutor)
  .addNode("tools", toolsNode)
  .addEdge(START, "tutor")
  .addConditionalEdges("tutor", route, ["tools", END])
  .addEdge("tools", "tutor")
  .compile({ checkpointer: new MemorySaver() });
await showGraph(agent);

// stream 으로 돌리며 State 를 받을 때마다 messages 개수와 마지막 메시지만 찍는다 (기록을 따라가는 게 목적이라 짧게)
async function run(graph: any, input: any, thread: any) {
  for await (const state of await graph.stream(input, { ...thread, streamMode: "values" })) {
    console.log(`State · messages ${state.messages.length}개 · 마지막 메시지:`);
    pretty(state.messages.at(-1));
    console.log();   // State 사이를 한 줄 띄운다
  }
}

## ① 지난 기록 — getStateHistory

먼저 한 번 끝까지 돌립니다.

In [ ]:
const thread1 = { configurable: { thread_id: "time-travel-1" } };
await run(agent, ask("ubiquitous 뜻을 사전에서 찾아 줘"), thread1);

`getStateHistory` 는 이 thread 의 checkpoint 를 **최근 것부터** 하나씩 돌려줍니다. 각각이 getState 로 보던 것과 같은 모양입니다(values · next · config).

In [ ]:
// ① 지난 기록: checkpoint 마다 단계 · 다음 노드 · messages 수를 찍는다
const history: any[] = [];
for await (const snapshot of agent.getStateHistory(thread1)) history.push(snapshot);
for (const s of history) {
  const last = s.values.messages?.at(-1);
  console.log(`step ${String(s.metadata.step).padStart(2)} · next ${JSON.stringify(s.next).padEnd(14)} · messages ${s.values.messages?.length ?? 0}개 · 마지막 ${last?.role ?? "-"}`);
}

**돌려 보면** 맨 아래 step -1 이 처음 넣은 입력, 맨 위가 지금입니다. 노드가 하나 돌 때마다 checkpoint 가 하나씩 쌓였습니다. `next` 를 보면 그 시점에 다음으로 돌 노드가 보입니다.

## ② 다시 돌리기 — replay

tutor 가 처음 모델을 부르기 **직전** checkpoint 를 골라, 그 config 로 `stream(null)` 을 부릅니다. config 에 `checkpoint_id` 가 들어 있어서 그 자리부터 돕니다.

In [ ]:
// ② 다시 돌리기: 처음 모델을 부르기 직전 checkpoint 에서 이어 간다
const beforeTutor = history.find((s) => s.next[0] === "tutor" && s.values.messages.length === 2);
console.log("고른 checkpoint:", beforeTutor.config.configurable.checkpoint_id);
await run(agent, null, beforeTutor.config);

**돌려 보면** 그 자리의 State(질문까지)가 먼저 오고, tutor → tools → tutor 가 다시 돕니다. tutor 는 **모델을 다시 부릅니다**. 답이 처음과 조금 다를 수 있는 이유입니다.

## ③ 갈라지기 — fork

이번에는 같은 checkpoint 에 `updateState` 로 학생의 말을 하나 더 넣습니다. 지난 checkpoint 를 고치는 게 아니라, 그 뒤에 **새 checkpoint 가 하나 생깁니다**. 돌려받은 config 로 이어 가면 새 갈래로 돕니다.

In [ ]:
// ③ 갈라지기: 지난 checkpoint 에 값을 넣으면 새 갈래가 생긴다
const forkConfig = await agent.updateState(beforeTutor.config, {
  messages: [{ role: "user", content: "아니, ubiquitous 말고 meticulous 를 찾아 줘" }],
});
await run(agent, null, forkConfig);

**돌려 보면** 모델은 고친 State 를 보고 meticulous 를 찾습니다. 앞의 기록은 지워지지 않았습니다. 기록이 몇 개인지 다시 세어 봅니다.

In [ ]:
let count = 0;
for await (const _ of agent.getStateHistory(thread1)) count++;
console.log(`처음 ${history.length}개 → 지금 ${count}개. 다시 돌리고 갈라진 checkpoint 가 같은 thread 에 쌓였다`);
console.log("지금 State 의 마지막 메시지:", (await agent.getState(thread1)).values.messages.at(-1).content);

**돌려 보면** checkpoint 가 늘었습니다. 다시 돌린 것도, 갈라진 것도 같은 thread 에 쌓입니다. `getState(thread1)` 은 가장 최근 것, 즉 갈라진 쪽의 답을 보여 줍니다.

오래 걸린 실행을 처음부터 다시 돌리지 않고 문제 난 자리 직전으로 돌아가 보거나, "그때 다르게 답했으면?" 을 확인할 때 씁니다.